In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import json
import joblib
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

DATA_DIR = "/content/drive/MyDrive/NADAC"
MODEL_PATH = "/content/drive/MyDrive/LightGBM/lightgbm_delta_model.pkl"
FEATURE_META_PATH = "/content/drive/MyDrive/LightGBM/feature_metadata.json"

FORECAST_DIR = "/content/drive/MyDrive/LightGBM/forecasts"
os.makedirs(FORECAST_DIR, exist_ok=True)

In [ ]:
model = joblib.load(MODEL_PATH)

with open(FEATURE_META_PATH, "r") as f:
    meta = json.load(f)

feat_cols = meta["features"]
LAGS = meta["lags"]

print("Model loaded.")

In [ ]:
def load_ndc_history(target_ndc):
    target_ndc = str(target_ndc).zfill(11)

    files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])
    rows = []

    for file in files:
        path = os.path.join(DATA_DIR, file)

        temp = pd.read_csv(
            path,
            usecols=["NDC", "NADAC Per Unit", "As of Date"],
            dtype={"NDC": str}
        )

        temp["NDC"] = temp["NDC"].str.zfill(11)
        temp = temp[temp["NDC"] == target_ndc]

        if not temp.empty:
            rows.append(temp)

    if not rows:
        raise ValueError("No data found")

    df = pd.concat(rows, ignore_index=True)

    df["As of Date"] = pd.to_datetime(df["As of Date"], errors="coerce")
    df["NADAC Per Unit"] = pd.to_numeric(df["NADAC Per Unit"], errors="coerce")

    df = df.dropna()

    df = df.drop_duplicates(
        subset=["NDC", "As of Date"],
        keep="last"
    )

    df["month"] = df["As of Date"].dt.to_period("M").dt.to_timestamp("M")

    monthly = (
        df.sort_values("As of Date")
        .groupby(["NDC", "month"], as_index=False)
        .last()
    )

    monthly = monthly[["month", "NADAC Per Unit"]]
    monthly = monthly.sort_values("month").reset_index(drop=True)

    return monthly

In [ ]:
def fill_missing(df):
    df = df.set_index("month")

    full_idx = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq="ME"
    )

    df = df.reindex(full_idx)

    df["NADAC Per Unit"] = df["NADAC Per Unit"].ffill()

    return df.reset_index().rename(columns={"index": "month"})

In [ ]:
def forecast_delta(model, history_df, steps=12):
    history_df = history_df.sort_values("month").reset_index(drop=True)

    if len(history_df) < 12:
        raise ValueError("Need at least 12 months history")

    known = {
        row["month"]: float(row["NADAC Per Unit"])
        for _, row in history_df.iterrows()
    }

    last_month = max(known.keys())

    future_months = pd.date_range(
        start=last_month + pd.offsets.MonthEnd(1),
        periods=steps,
        freq="ME"
    )

    results = []

    for target_month in future_months:
        feature_dict = {}

        # lag features
        for k in LAGS:
            lag_month = target_month - pd.offsets.MonthEnd(k)
            feature_dict[f"lag_{k}"] = known[lag_month]

        # rolling
        vals_3 = [
            known[target_month - pd.offsets.MonthEnd(k)]
            for k in range(1, 4)
        ]

        vals_6 = [
            known[target_month - pd.offsets.MonthEnd(k)]
            for k in range(1, 7)
        ]

        feature_dict["roll_mean_3"] = float(np.mean(vals_3))
        feature_dict["roll_mean_6"] = float(np.mean(vals_6))
        feature_dict["roll_std_6"] = float(np.std(vals_6, ddof=1))

        # month
        feature_dict["month_num"] = target_month.month

        # predict delta
        X_pred = pd.DataFrame([feature_dict], columns=feat_cols)
        delta = float(model.predict(X_pred)[0])

        prev_month = target_month - pd.offsets.MonthEnd(1)
        last_price = known[prev_month]

        pred_price = last_price + delta

        results.append({
            "month": target_month.strftime("%Y-%m"),
            "predicted_price": pred_price
        })

        # recursive
        known[target_month] = pred_price

    return results

In [ ]:
TARGET_NDC = "00054006447"

hist = load_ndc_history(TARGET_NDC)
hist = fill_missing(hist)

forecast = forecast_delta(model, hist, steps=12)

forecast